# Stock Market Direction Prediction — Support Vector Machine

**Student ID:** IT25102165  
**Model:** Linear Support Vector Machine (SVM)  
**Task:** Predict whether the next closing price moves up (`1`) or down/not up (`0`).

A linear SVM is used because it is efficient for the full 77,150-row training set. Model selection uses training and validation data. The test set is reserved for final evaluation.

## 1. Import libraries and configure the experiment

In [ ]:
from pathlib import Path
import json
import shutil
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
OUTPUT_DIR = Path('IT25102165_SVM_Results')
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)
print('Setup completed.')

## 2. Upload the three selected datasets

In [ ]:
required_files = ['train_selected.csv', 'validation_selected.csv', 'test_selected.csv']
missing_files = [name for name in required_files if not Path(name).exists()]
if missing_files:
    try:
        from google.colab import files
        print('Upload:', ', '.join(missing_files))
        files.upload()
    except ImportError:
        print('Place the three CSV files beside this notebook.')
still_missing = [name for name in required_files if not Path(name).exists()]
if still_missing:
    raise FileNotFoundError(f'Missing files: {still_missing}')
print('All required datasets are available.')

## 3. Load and validate the datasets

In [ ]:
train_df = pd.read_csv('train_selected.csv')
val_df = pd.read_csv('validation_selected.csv')
test_df = pd.read_csv('test_selected.csv')
datasets = {'Training': train_df, 'Validation': val_df, 'Testing': test_df}
expected_columns = list(train_df.columns)
for name, df in datasets.items():
    assert list(df.columns) == expected_columns, f'{name} columns do not match.'
    assert 'Target' in df.columns, f'Target is missing from {name}.'
    assert df.isna().sum().sum() == 0, f'{name} contains missing values.'
    assert set(df['Target'].unique()).issubset({0, 1}), f'{name} Target must be binary.'
    print(f'{name:10s}: {df.shape[0]:,} rows, {df.shape[1]} columns')
display(train_df.head())

## 4. Examine target distribution

In [ ]:
rows = []
for name, df in datasets.items():
    counts = df['Target'].value_counts().sort_index()
    rows.append({'Dataset': name, 'Down_or_Not_Up_0': int(counts.get(0, 0)),
                 'Up_1': int(counts.get(1, 0)),
                 'Up_Percentage': round(100 * (df['Target'] == 1).mean(), 2)})
target_distribution = pd.DataFrame(rows)
display(target_distribution)
target_distribution.to_csv(OUTPUT_DIR / 'target_distribution.csv', index=False)
ax = target_distribution.set_index('Dataset')[['Down_or_Not_Up_0', 'Up_1']].plot(
    kind='bar', figsize=(8, 5), color=['#d95f5f', '#4c9f70'])
ax.set(title='Target Distribution by Dataset', xlabel='Dataset', ylabel='Number of records')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / '01_target_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Separate features and target

In [ ]:
FEATURES = [c for c in train_df.columns if c != 'Target']
X_train, y_train = train_df[FEATURES], train_df['Target']
X_val, y_val = val_df[FEATURES], val_df['Target']
X_test, y_test = test_df[FEATURES], test_df['Target']
print('Features:', FEATURES)
print('Training:', X_train.shape, '| Validation:', X_val.shape, '| Testing:', X_test.shape)

## 6. Train the baseline Linear SVM

SVM finds a decision boundary with the largest possible margin between classes. `LinearSVC` is appropriate here because the dataset is large and the inputs were already scaled by the group pipeline.

In [ ]:
def calculate_metrics(y_true, y_pred, decision_scores):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1_Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, decision_scores)
    }

baseline_model = LinearSVC(C=1.0, class_weight=None, max_iter=10000,
                           dual='auto', random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_val_pred = baseline_model.predict(X_val)
baseline_val_scores = baseline_model.decision_function(X_val)
baseline_metrics = calculate_metrics(y_val, baseline_val_pred, baseline_val_scores)
display(pd.DataFrame([baseline_metrics]).round(4))
print(classification_report(y_val, baseline_val_pred, target_names=['Down/Not Up', 'Up']))

## 7. Perform 5-fold cross-validation on training data

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_result = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
cv_scores = pd.DataFrame({
    'Fold': np.arange(1, 6), 'Accuracy': cv_result['test_accuracy'],
    'Precision': cv_result['test_precision'], 'Recall': cv_result['test_recall'],
    'F1_Score': cv_result['test_f1'], 'ROC_AUC': cv_result['test_roc_auc']})
display(cv_scores.round(4))
display(cv_scores.drop(columns='Fold').agg(['mean', 'std']).round(4))
cv_scores.to_csv(OUTPUT_DIR / 'cross_validation_scores.csv', index=False)

## 8. Tune SVM hyperparameters

`C` controls the trade-off between a wider margin and training errors. Class weighting tests whether balancing the classes improves performance. Tuning uses only training data.

In [ ]:
parameter_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10],
    'class_weight': [None, 'balanced'],
    'loss': ['hinge', 'squared_hinge']
}
grid_search = GridSearchCV(
    LinearSVC(max_iter=10000, dual='auto', random_state=RANDOM_STATE),
    parameter_grid, scoring='f1', cv=cv, n_jobs=-1, refit=True, return_train_score=True)
grid_search.fit(X_train, y_train)
print('Best parameters:', grid_search.best_params_)
print('Best mean CV F1 score:', round(grid_search.best_score_, 4))
tuning_results = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
tuning_results.to_csv(OUTPUT_DIR / 'hyperparameter_tuning_results.csv', index=False)
display(tuning_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

## 9. Compare baseline and tuned models on validation data

In [ ]:
tuned_model = grid_search.best_estimator_
tuned_val_pred = tuned_model.predict(X_val)
tuned_val_scores = tuned_model.decision_function(X_val)
tuned_metrics = calculate_metrics(y_val, tuned_val_pred, tuned_val_scores)
validation_comparison = pd.DataFrame([
    {'Model': 'Baseline Linear SVM', **baseline_metrics},
    {'Model': 'Tuned Linear SVM', **tuned_metrics}])
display(validation_comparison.round(4))
validation_comparison.to_csv(OUTPUT_DIR / 'validation_model_comparison.csv', index=False)
ax = validation_comparison.set_index('Model').plot(kind='bar', figsize=(10, 5), ylim=(0, 1), colormap='plasma')
ax.set(title='Baseline vs Tuned SVM — Validation Set', ylabel='Score')
ax.tick_params(axis='x', rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PLOT_DIR / '02_validation_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Train final SVM and evaluate the untouched test set

In [ ]:
X_train_final = pd.concat([X_train, X_val], ignore_index=True)
y_train_final = pd.concat([y_train, y_val], ignore_index=True)
final_model = LinearSVC(**grid_search.best_params_, max_iter=10000,
                        dual='auto', random_state=RANDOM_STATE)
final_model.fit(X_train_final, y_train_final)
test_pred = final_model.predict(X_test)
test_scores = final_model.decision_function(X_test)
test_metrics = calculate_metrics(y_test, test_pred, test_scores)
print('Final test metrics')
display(pd.DataFrame([test_metrics]).round(4))
print(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'final_test_metrics.csv', index=False)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as file:
    file.write(classification_report(y_test, test_pred, target_names=['Down/Not Up', 'Up']))

## 11. Visualize the final evaluation

In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False,
            xticklabels=['Down/Not Up', 'Up'], yticklabels=['Down/Not Up', 'Up'], ax=axes[0])
axes[0].set(title='Confusion Matrix', xlabel='Predicted class', ylabel='Actual class')
fpr, tpr, _ = roc_curve(y_test, test_scores)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['ROC_AUC']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set(title='ROC Curve', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[1].legend()
pr_precision, pr_recall, _ = precision_recall_curve(y_test, test_scores)
axes[2].plot(pr_recall, pr_precision, color='#c51b7d')
axes[2].set(title='Precision–Recall Curve', xlabel='Recall', ylabel='Precision')
plt.tight_layout()
plt.savefig(PLOT_DIR / '03_final_evaluation_curves.png', dpi=200, bbox_inches='tight')
plt.show()

## 12. Interpret SVM feature weights

In [ ]:
weight_table = pd.DataFrame({'Feature': FEATURES, 'Weight': final_model.coef_[0]})
weight_table['Absolute_Weight'] = weight_table['Weight'].abs()
weight_table = weight_table.sort_values('Absolute_Weight', ascending=False)
display(weight_table)
weight_table.to_csv(OUTPUT_DIR / 'feature_weights.csv', index=False)
plot_weights = weight_table.sort_values('Weight')
colors = ['#d95f5f' if value < 0 else '#4c9f70' for value in plot_weights['Weight']]
plt.figure(figsize=(9, 5))
plt.barh(plot_weights['Feature'], plot_weights['Weight'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear SVM Feature Weights')
plt.xlabel('Weight')
plt.tight_layout()
plt.savefig(PLOT_DIR / '04_feature_weights.png', dpi=200, bbox_inches='tight')
plt.show()

## 13. Save predictions, trained model and experiment summary

In [ ]:
prediction_results = X_test.copy()
prediction_results['Actual_Target'] = y_test.to_numpy()
prediction_results['Predicted_Target'] = test_pred
prediction_results['Decision_Score'] = test_scores
prediction_results['Prediction_Correct'] = prediction_results['Actual_Target'] == prediction_results['Predicted_Target']
prediction_results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
display(prediction_results.head(15))
joblib.dump(final_model, OUTPUT_DIR / 'IT25102165_svm_model.joblib')

summary = {
    'student_id': 'IT25102165', 'algorithm': 'Linear Support Vector Machine',
    'problem_type': 'Binary classification',
    'target_definition': '1 if next closing price is higher; otherwise 0',
    'features': FEATURES, 'training_rows': int(len(train_df)),
    'validation_rows': int(len(val_df)), 'testing_rows': int(len(test_df)),
    'best_parameters': grid_search.best_params_,
    'best_cross_validation_f1': float(grid_search.best_score_),
    'baseline_validation_metrics': {k: float(v) for k, v in baseline_metrics.items()},
    'tuned_validation_metrics': {k: float(v) for k, v in tuned_metrics.items()},
    'final_test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'limitations': [
        'A linear boundary may not represent complex market relationships.',
        'Historical patterns do not guarantee future performance.',
        'The selected features contain limited short-term market information.',
        'News, transaction costs and macroeconomic variables are excluded.'
    ]
}
with open(OUTPUT_DIR / 'experiment_summary.json', 'w') as file:
    json.dump(summary, file, indent=2)
print(json.dumps(summary, indent=2))

## 14. Conclusion

After running the notebook, report the final Accuracy, Precision, Recall, F1 and ROC-AUC. Compare the baseline and tuned validation scores, explain the effect of `C` and class weighting, and state whether the model performs better than predicting the majority class.

## 15. Create and download the results ZIP

In [ ]:
zip_path = shutil.make_archive('IT25102165_SVM_Results', 'zip', OUTPUT_DIR)
print('Created:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Download the ZIP from the notebook folder:', zip_path)